# 02 · Memahami GT publik: apa itu `D`, kapan `H`/`A` berganti

**Tujuan.** CHALLENGE.md: *"the question that matters is not how the labels are computed, but what the corrected
tracking means by 'dead' and by 'owning' — and the public ground truth is where you can see it."* Notebook ini
membaca 5 menit GT publik untuk memahami **definisi**, lalu menerjemahkannya menjadi keputusan desain.

| | |
|---|---|
| **Input** | `cache/prep.npz`, `cache/tags.csv`, GT publik (`external/.../BAR-ATH_possession_gt_public.csv`) |
| **Output** | laporan `cache/reports/02_eda_gt_publik.*` (tidak ada artefak yang dipakai tahap lain) |
| **Waktu** | < 1 menit |

> ⚠️ **5 menit = 21 segmen, 17 possession, 4 bola mati.** Semua angka di sini *eksploratif*: dipakai untuk
> memahami definisi dan memilih **jenis** fitur, **bukan** untuk menyetel ambang atau bobot. Tidak ada parameter
> pipeline yang diambil dari notebook ini (lihat §5 di `docs/RENCANA.md`).

| Bagian | Pertanyaan |
|---|---|
| 1 | Seperti apa struktur GT publik? |
| 2 | Bagaimana bola dan pemain berperilaku saat `D`? Fitur apa yang membedakan `D` dari bola hidup? |
| 3 | Fitur apa yang membedakan possession `H` dari `A` — termasuk saat bola tidak terlihat? |
| 4 | Apa yang terjadi di sekitar setiap pergantian label? Seberapa cepat "pemain terdekat" mengikuti? |
| 5 | Seberapa cocok tim di Smart Tagging dengan tim yang menguasai bola menurut GT? |
| 6 | Restart: siapa yang memulai setelah `D`, dan apa kata tag? |

In [ ]:
%run ./common.ipynb
import warnings

## Parameter

Hanya parameter *analisis* (lebar jendela untuk meringkas), bukan parameter pipeline.

| Parameter | Nilai | Arti |
|---|---|---|
| `CTX` | 125 frame (5 s) | konteks di kiri/kanan segmen `D` pada garis waktu |
| `NEAR_WIN` | 50 frame (2 s) | jendela sebelum/sesudah transisi untuk meringkas "tim terdekat" |
| `LAG_SEARCH` | 75 frame (3 s) | rentang pencarian kapan tim terdekat berpindah ke tim baru |
| `SMOOTH` | 101 frame (± 2 s) | perataan fitur bentuk tim |

In [ ]:
CTX = 125
NEAR_WIN = 50
LAG_SEARCH = 75
SMOOTH = 101

In [ ]:
rep = Report("02_eda_gt_publik")
z = load_prep()
tags = load_tags()
P, HOME = z["players"], z["slot_is_home"]
obs = z["ball_src"] > 0
bx, by = z["ball_filled"][:, 0], z["ball_filled"][:, 1]
adir = z["attack_dir"].astype(np.float32)

gt = pm.read_label_csv(str(GT_PUBLIC))["label"].astype(str).str.strip().str.upper().to_numpy()
known = np.flatnonzero(gt != "X")
W0, W1 = int(known[0]), int(known[-1]) + 1
WIN = np.zeros(N_FRAMES, dtype=bool)
WIN[W0:W1] = True
isH, isA, isD = WIN & (gt == "H"), WIN & (gt == "A"), WIN & (gt == "D")
live = isH | isA

## 1. Struktur GT publik

Possession = run maksimal `H` atau `A`; `D` adalah jeda, bukan possession (CHALLENGE.md). Tabel segmen ini
publik (ikut di repo resmi), jadi aman ditampilkan.

In [ ]:
seg = segments_of(gt[W0:W1], W0)
poss = seg[seg.label != "D"]
rep("jendela {}..{} ({} frame, {:.1f} menit) | segmen {} | possession {} | D {}".format(
    W0, W1 - 1, W1 - W0, (W1 - W0) / FPS / 60, len(seg), len(poss), int((seg.label == "D").sum())))
rep("durasi possession: rata-rata {:.1f} s | median {:.1f} s | min {:.1f} s | maks {:.1f} s".format(
    poss.dur_s.mean(), poss.dur_s.median(), poss.dur_s.min(), poss.dur_s.max()))
rep("durasi D: {}".format(", ".join("{:.1f} s".format(d) for d in seg[seg.label == "D"].dur_s)))
pairs = pd.Series(["{}→{}".format(a, b) for a, b in zip(seg.label[:-1], seg.label[1:])]).value_counts()
rep("jenis pergantian: {}".format(pairs.to_dict()))
seg

## 2. Bola mati (`D`)

### 2a. Fitur per frame

| Fitur | Definisi |
|---|---|
| `obs` | bola terlihat (teramati atau diinterpolasi) |
| `ball_v` | kecepatan bola dari selisih posisi ± 12 frame (lebih halus dari antar frame) |
| `ball_sd` | simpangan posisi bola dalam ± 0,5 s — kecil = bola diam |
| `line_m` | jarak bola ke garis lapangan terdekat |
| `oob_m` | jarak bola di luar lapangan |
| `p_speed` | kecepatan rata-rata semua pemain terlacak, dirata ± 1 s — pemain melambat saat bola mati |

In [ ]:
h = 12
ball_v = np.full(N_FRAMES, np.nan)
ball_v[h:-h] = np.hypot(bx[2 * h:] - bx[:-2 * h], by[2 * h:] - by[:-2 * h]) / (2 * h / FPS)
xo = pd.Series(np.where(obs, bx, np.nan))
yo = pd.Series(np.where(obs, by, np.nan))
ball_sd = np.hypot(xo.rolling(25, center=True, min_periods=15).std(),
                   yo.rolling(25, center=True, min_periods=15).std()).to_numpy()
line_m = np.where(obs, np.minimum.reduce([bx, PITCH_L - bx, by, PITCH_W - by]), np.nan)
oob_m = np.nan_to_num(z["ball_oob_m"], nan=0.0)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    p_speed = pd.Series(np.nanmean(P[..., 2], axis=1)).rolling(50, center=True, min_periods=10).mean().to_numpy()

### 2b. Garis waktu per detik di sekitar setiap segmen `D`

Satu huruf = satu detik; bagian tengah (`D: …`) adalah segmen D, kiri/kanan 5 detik konteks.

`.` bola hilang (> 50% detik itu) · `O` bola di luar garis · `s` diam (sd < 0,25 m) · `w` pelan (< 2 m/s) ·
`r` berjalan (2–6 m/s) · `F` cepat (> 6 m/s) · `?` tak terukur

Baca polanya dengan kata-kata: bagaimana sebuah `D` dimulai, apa yang terjadi di dalamnya, bagaimana ia berakhir.

In [ ]:
def second_state(a, b):
    if obs[a:b].mean() < 0.5:
        return "."
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        if np.nanmedian(oob_m[a:b]) > 0.5:
            return "O"
        if np.nanmedian(ball_sd[a:b]) < 0.25:
            return "s"
        v = np.nanmedian(ball_v[a:b])
    if not np.isfinite(v):
        return "?"
    return "w" if v < 2 else ("r" if v < 6 else "F")


d_segs = seg[seg.label == "D"].reset_index(drop=True)
for i, r in d_segs.iterrows():
    pre = "".join(second_state(t, t + FPS) for t in range(r.start - CTX, r.start, FPS))
    mid = "".join(second_state(t, min(t + FPS, r.end)) for t in range(r.start, r.end, FPS))
    post = "".join(second_state(t, t + FPS) for t in range(r.end, r.end + CTX, FPS))
    rep("D{} ({:5.1f} s): {} | D: {} | {}".format(i + 1, r.dur_s, pre, mid, post))

### 2c. Bukti di dalam tiap segmen `D` dibanding bola hidup (% frame)

In [ ]:
evidence = [("bola terlihat", lambda m: obs[m]),
            ("di luar garis >0,5 m", lambda m: oob_m[m] > 0.5),
            ("diam sd<0,5", lambda m: ball_sd[m] < 0.5),
            ("pelan v<2", lambda m: ball_v[m] < 2.0),
            ("dekat garis <2 m", lambda m: line_m[m] < 2.0)]


def evidence_row(m):
    row = {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        for name, f in evidence:
            row[name] = round(100 * float(np.mean(f(m))), 0)
        row["pemain m/s (median)"] = round(float(np.nanmedian(p_speed[m])), 2)
    return row


ev = {}
for i, r in d_segs.iterrows():
    m = np.zeros(N_FRAMES, dtype=bool)
    m[r.start:r.end] = True
    ev["D{}".format(i + 1)] = evidence_row(m)
ev["hidup (H/A)"] = evidence_row(live)
ev_df = pd.DataFrame(ev).T
rep.table(ev_df, "bukti per segmen D vs hidup (% frame):")

### 2d. Daya beda tiap fitur: `D` vs hidup (AUC)

AUC = peluang frame `D` acak punya skor lebih tinggi dari frame hidup acak. 0,5 = tak informatif; > 0,7 berguna.
Skor diberi tanda sehingga "lebih tinggi = lebih mirip bola mati".

In [ ]:
d_feats = {"bola hilang": (~obs).astype(float), "bola pelan (−v)": -ball_v, "bola diam (−sd)": -ball_sd,
           "dekat garis (−jarak)": -line_m, "di luar garis": oob_m, "pemain tenang (−v pemain)": -p_speed}
auc_d = pd.DataFrame([{"fitur": k, "AUC D vs hidup": round(auc(v[isD], v[live]), 3),
                       "cakupan D %": round(100 * np.isfinite(v[isD]).mean(), 0)} for k, v in d_feats.items()])
rep.table(auc_d.set_index("fitur"), "AUC fitur bola mati:")

## 3. Fitur yang membedakan possession `H` dari `A`

Fitur dihitung per tim lalu diambil **selisih H − A**, sehingga skor tinggi = mendukung `H`. Arah serang
memakai `attack_dir` (terverifikasi di notebook 01): untuk H tandanya `attack_dir`, untuk A kebalikannya.

| Fitur tim | Definisi | Dugaan untuk tim yang menguasai bola |
|---|---|---|
| `d_ball` | jarak pemain terdekat tim itu ke bola (hanya bila bola terlihat) | lebih dekat |
| `advance` | sentroid x tim relatif garis tengah, ke arah gawang lawan | lebih maju |
| `spread` | √(var x + var y) posisi pemain | lebih melebar |
| `width`, `depth` | rentang y dan rentang x tim | lebih lebar / lebih panjang |
| `v_goal` | rata-rata kecepatan pemain ke arah gawang lawan (per slot, ± 0,4 s) | lebih maju bergerak |
| `speed` | kecepatan rata-rata pemain | ? |
| bola di separuh lawan | (x bola − tengah) × arah serang H | bola di depan tim yang menguasai |

Bentuk tim dirata ± 2 s (`SMOOTH`), karena bentuk berubah lambat dan per frame berisik.

In [ ]:
def team_features(mask, sign):
    X, Y, S = P[:, mask, 0], P[:, mask, 1], P[:, mask, 2]
    k = 10
    with warnings.catch_warnings(), np.errstate(all="ignore"):
        warnings.simplefilter("ignore", category=RuntimeWarning)
        cx = np.nanmean(X, axis=1)
        vx = np.full(X.shape, np.nan, dtype=np.float32)
        vx[k:-k] = (X[2 * k:] - X[:-2 * k]) / (2 * k / FPS)
        return {
            "d_ball": np.where(obs, np.nanmin(np.hypot(X - bx[:, None], Y - by[:, None]), axis=1), np.nan),
            "advance": (cx - PITCH_L / 2) * sign,
            "spread": np.sqrt(np.nanvar(X, axis=1) + np.nanvar(Y, axis=1)),
            "width": np.nanmax(Y, axis=1) - np.nanmin(Y, axis=1),
            "depth": np.nanmax(X, axis=1) - np.nanmin(X, axis=1),
            "v_goal": np.nanmean(vx, axis=1) * sign,
            "speed": np.nanmean(S, axis=1),
        }


with timer("fitur tim"):
    fH = team_features(HOME, adir)
    fA = team_features(~HOME, -adir)


def smooth(v):
    return pd.Series(v).rolling(SMOOTH, center=True, min_periods=SMOOTH // 4).mean().to_numpy()


ha_feats = {
    "pemain terdekat ke bola (d_A − d_H)": fA["d_ball"] - fH["d_ball"],
    "bola di separuh lawan H": np.where(obs, (bx - PITCH_L / 2) * adir, np.nan),
}
for name in ("advance", "spread", "width", "depth", "v_goal", "speed"):
    diff = fH[name] - fA[name]
    ha_feats["{} H−A (per frame)".format(name)] = diff
    ha_feats["{} H−A (±2 s)".format(name)] = smooth(diff)

ball_hidden = live & ~obs
rows = []
for name, v in ha_feats.items():
    rows.append({"fitur": name,
                 "AUC semua frame hidup": round(auc(v[isH], v[isA]), 3),
                 "AUC saat bola hilang": round(auc(v[isH & ~obs], v[isA & ~obs]), 3),
                 "cakupan %": round(100 * np.isfinite(v[live]).mean(), 0)})
auc_ha = pd.DataFrame(rows).set_index("fitur")
rep("frame hidup {} | bola hilang di {} di antaranya ({:.0f}%)".format(
    int(live.sum()), int(ball_hidden.sum()), 100 * ball_hidden.sum() / live.sum()))
rep.table(auc_ha, "AUC H vs A (>0,5 = skor tinggi mendukung H; <0,5 = tandanya terbalik):")

### 3b. Kemurnian "tim terdekat" di dalam tiap possession

Untuk setiap possession GT: berapa persen frame bola terlihat, dan dari frame itu berapa persen pemain terdekat ke
bola berasal dari tim yang menguasai. Kemurnian rendah = pemain lawan sering paling dekat ke bola walau timnya
tidak menguasai (pressing, bola lepas) — itu sebabnya "pemain terdekat" per frame berkedip.

In [ ]:
near = np.where(obs, np.where(fH["d_ball"] < fA["d_ball"], 1, -1), 0)   # 1 = H terdekat, −1 = A, 0 = bola hilang
pur = []
for r in poss.itertuples():
    nn = near[r.start:r.end]
    own = 1 if r.label == "H" else -1
    vis = nn != 0
    pur.append({"label": r.label, "dur_s": round(r.dur_s, 1), "bola terlihat %": round(100 * vis.mean(), 0),
                "tim sendiri terdekat %": round(100 * (nn[vis] == own).mean(), 0) if vis.any() else np.nan})
pur = pd.DataFrame(pur)
rep("kemurnian tim terdekat per possession: median {:.0f}% | min {:.0f}% | possession dengan bola terlihat < 50%: {}".format(
    pur["tim sendiri terdekat %"].median(), pur["tim sendiri terdekat %"].min(), int((pur["bola terlihat %"] < 50).sum())))
pur

## 4. Di sekitar setiap pergantian label

Untuk setiap batas segmen GT pada frame *t*:

| Kolom | Arti |
|---|---|
| `bola ±1s %` | persentase frame dengan bola terlihat di [t−25, t+25) |
| `baru terdekat sebelum/sesudah %` | dari frame bola terlihat di 2 s sebelum/sesudah *t*, berapa persen pemain terdekat dari tim **baru** |
| `lag (frame)` | frame pertama di [t−3 s, t+3 s] di mana ≥ 80% (dari ≥ 10 frame terlihat) pada 1 s berikutnya tim baru terdekat, dikurangi *t*. Negatif = data sudah menunjukkan tim baru **sebelum** GT berganti |
| `jarak tim baru (m)` | median jarak pemain terdekat tim baru ke bola di 0,5 s pertama |

Ini menjawab: apakah GT mengganti possession saat tim baru **menyentuh** bola, atau setelah **menguasai** beberapa saat?

In [ ]:
def share(a, b, team):
    nn = near[max(a, 0):b]
    vis = nn != 0
    return round(100 * float((nn[vis] == team).mean()), 0) if vis.sum() >= 5 else np.nan


trans = []
for i in range(1, len(seg)):
    t, frm, to = int(seg.start[i]), seg.label[i - 1], seg.label[i]
    row = {"frame": t, "jenis": "{}→{}".format(frm, to), "bola ±1s %": round(100 * obs[t - 25:t + 25].mean(), 0)}
    if to in ("H", "A"):
        new = 1 if to == "H" else -1
        row["baru terdekat sebelum %"] = share(t - NEAR_WIN, t, new)
        row["baru terdekat sesudah %"] = share(t, t + NEAR_WIN, new)
        lag = np.nan
        for f in range(t - LAG_SEARCH, t + LAG_SEARCH + 1):
            nn = near[f:f + FPS]
            vis = nn != 0
            if vis.sum() >= 10 and (nn[vis] == new).mean() >= 0.8:
                lag = f - t
                break
        row["lag (frame)"] = lag
        d_new = (fH if to == "H" else fA)["d_ball"][t:t + 13]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            row["jarak tim baru (m)"] = round(float(np.nanmedian(d_new)), 1)
    trans.append(row)
trans = pd.DataFrame(trans)

for kind, sel in (("pergantian H↔A", trans.jenis.isin(["H→A", "A→H"])), ("restart D→H/A", trans.jenis.str.startswith("D"))):
    t_ = trans[sel]
    rep("{}: n {} | bola terlihat ±1 s median {:.0f}% | tim baru terdekat sebelum {:.0f}% → sesudah {:.0f}% (median) | "
        "lag median {} frame (n terukur {}), |lag| ≤ 12: {}".format(
            kind, len(t_), t_["bola ±1s %"].median(), t_["baru terdekat sebelum %"].median(),
            t_["baru terdekat sesudah %"].median(), t_["lag (frame)"].median(), int(t_["lag (frame)"].notna().sum()),
            int((t_["lag (frame)"].abs() <= 12).sum())))
trans

## 5. Smart Tagging vs GT

Kode tag dikelompokkan menurut **siapa yang menguasai bola** menurut definisinya (OVERVIEW.md):

- **menyerang** — `Team` = tim yang menguasai: Build Up, Progression, Creating Chances, Attacking Transition,
  Ball in Final Third, Ball in the Box, Players in the Box, Long Ball
- **bertahan** — `Team` = tim yang **tidak** menguasai: Defending in Attacking/Middle/Defensive Third,
  Defensive Transition

Untuk setiap tag yang beririsan dengan jendela publik: dari frame hidup di dalam jendela tag, berapa persen GT
sama dengan tim yang tersirat. Sampelnya kecil; hasil ini menentukan apakah tag layak jadi fitur dan proxy metric.

In [ ]:
ATTACK_CODES = {"BUILD UP", "PROGRESSION", "CREATING CHANCES", "ATTACKING TRANSITION", "BALL IN FINAL THIRD",
                "BALL IN THE BOX", "PLAYERS IN THE BOX", "LONG BALL"}
DEFEND_CODES = {"DEFENDING IN ATTACKING THIRD", "DEFENDING IN MIDDLE THIRD", "DEFENDING IN DEFENSIVE THIRD",
                "DEFENSIVE TRANSITION"}
other = {"H": "A", "A": "H"}
tag_rows = []
for r in tags.itertuples():
    if r.side not in ("H", "A") or (r.code not in ATTACK_CODES and r.code not in DEFEND_CODES):
        continue
    a, b = max(r.start_f, W0), min(r.end_f, W1)
    if a >= b or not live[a:b].any():
        continue
    implied = r.side if r.code in ATTACK_CODES else other[r.side]
    g = gt[a:b][live[a:b]]
    tag_rows.append({"code": r.code, "kelompok": "menyerang" if r.code in ATTACK_CODES else "bertahan",
                     "frame hidup": len(g), "cocok %": 100 * float((g == implied).mean())})
tag_agree = pd.DataFrame(tag_rows)
if len(tag_agree):
    by_group = tag_agree.groupby("kelompok").apply(
        lambda d: pd.Series({"tag": len(d), "frame hidup": int(d["frame hidup"].sum()),
                             "cocok % (tertimbang frame)": round(float(np.average(d["cocok %"], weights=d["frame hidup"])), 1),
                             "tag dengan cocok ≥ 80%": int((d["cocok %"] >= 80).sum())}), include_groups=False)
    rep.table(by_group, "kecocokan tag vs GT di jendela publik:")
    by_code = tag_agree.groupby("code").agg(tag=("cocok %", "size"), frame=("frame hidup", "sum"),
                                            cocok_pct=("cocok %", "mean")).round(1)
    rep.table(by_code, "per kode:")

## 6. Restart setelah `D`

Untuk setiap segmen `D` di GT: label sesudahnya, tag `SET PIECES` terdekat (jenis & tim), dan siapa yang paling
dekat ke bola di 2 s pertama setelah restart.

In [ ]:
sp = tags[tags.code == "SET PIECES"]
rs = []
for i, r in d_segs.iterrows():
    nxt = gt[r.end] if r.end < W1 else "?"
    cand = sp[(sp.start_f <= r.end + 250) & (sp.end_f >= r.start - 250)]
    if len(cand):
        c = cand.iloc[(cand.start_f - r.end).abs().argmin()]
        tag_txt = "{} / {} (mulai {:+.1f} s dari restart)".format(c.Type, c.side, (c.start_f - r.end) / FPS)
    else:
        tag_txt = "-"
    new = 1 if nxt == "H" else -1
    rs.append({"D": "D{}".format(i + 1), "dur_s": round(r.dur_s, 1), "restart oleh": nxt, "tag set piece": tag_txt,
               "tim restart terdekat 2 s %": share(r.end, r.end + NEAR_WIN, new)})
rs = pd.DataFrame(rs)
rep.table(rs.set_index("D"), "restart:")

In [ ]:
rep.save({"segments": len(seg), "possessions": len(poss), "auc_dead": auc_d.to_dict("records"),
          "auc_ha": auc_ha.reset_index().to_dict("records"), "purity": pur.to_dict("records"),
          "transitions": trans.to_dict("records"),
          "tag_agreement": tag_agree.to_dict("records") if len(tag_agree) else [], "restarts": rs.to_dict("records")})

## Ringkasan & keputusan

> Sampel: 5 menit, 17 possession, 4 `D`, 12 pergantian H↔A. Temuan di bawah dipakai untuk memilih **jenis**
> fitur dan **struktur** model, bukan angka parameter.

### Bola mati (`D`)

1. **Sinyal terkuat adalah pemain melambat** (AUC ± 0,84); bola dekat garis ± 0,77; bola diam/pelan ± 0,71.
   Bola di luar garis hampir tak berguna (0,54) — tracker kehilangan bola justru saat bola keluar.
2. **Bola hidup hampir selalu terlihat** di jendela ini (± 94% frame hidup), sedangkan dua dari empat `D`
   bolanya hampir tak terlihat sama sekali. Karena di seluruh match bola hanya terlihat ± 59% frame dinilai dan
   hampir semua frame hilang ada di celah > 5 s (notebook 01), **celah bola yang panjang kemungkinan besar
   sebagian besar adalah bola mati**. → fitur "bola hilang lama" layak masuk detektor `D`.
3. Dua `D` lain bolanya terlihat penuh (bola dibawa/diletakkan untuk restart) → detektor tidak boleh hanya
   mengandalkan bola hilang; pola gerak pemain tetap inti.

### Siapa yang menguasai (`H`/`A`)

4. **Bentuk tim adalah sinyal terkuat, dan tetap ada saat bola hilang.** Tim yang menguasai bola **lebih lebar**
   (selisih lebar H−A: AUC ± 0,95) dan **lebih menyebar** (± 0,91), serta **lebih lambat** dari lawannya
   (selisih kecepatan: AUC ± 0,14, artinya tandanya terbalik → ± 0,86 untuk "tim lebih lambat menguasai").
   Tim yang bertahan menekan dan mengejar; tim yang menguasai melebar dan mengatur.
5. **Pemain terdekat ke bola** informatif (AUC ± 0,85) tetapi berkedip: rata-rata hanya ± 74% frame sebuah
   possession tim sendiri yang terdekat. Wajar — lawan menekan dari dekat.
6. **Tidak informatif:** sentroid tim yang lebih maju (± 0,48), posisi bola di separuh lawan.

→ **Keputusan Fase 2:** model H/A menggabungkan *pemain terdekat* (bila bola terlihat) dengan *bentuk tim*
(lebar, sebaran, kedalaman, kecepatan relatif, gerak ke gawang lawan) yang selalu tersedia. Bentuk tim dapat
dilatih dari pseudo-label "kontrol jelas" seluruh match tanpa menyentuh GT — sesuai rencana.

### Waktu pergantian

7. **Pergantian H↔A tidak seragam.** Pada sebagian besar pergantian, tim baru sudah menjadi pemain terdekat
   ± 1–3 s **sebelum** GT berganti (masa perebutan/duel tetap milik tim lama — sesuai CHALLENGE.md: *"in phases where
   the ball is contested or loose, possession stays with the last team"*). Pada beberapa lainnya tim baru baru
   terdekat **setelah** GT berganti (umpan yang dipotong), atau tidak pernah (possession sangat pendek).
   → batas possession **tidak** bisa diambil dari "pertama kali tim baru terdekat"; perlu aturan *kontrol*
   (tim baru terdekat dan bertahan) dan penyempurnaan batas di Fase 2.7.
8. **GT memuat possession sangat pendek** (6 dari 17 di bawah 4 s; terpendek 0,8 s). spellF1 menghitungnya
   sama dengan possession panjang. → durasi minimum possession di decoder harus kecil, dan
   *trade-off*-nya dengan possession palsu harus diukur, bukan ditebak.

### Restart

9. **Tim yang melakukan restart adalah tim terdekat ke bola selama akhir segmen `D`** (4 dari 4, ± 100% frame
   terlihat). → setelah `D`, possession pertama = tim terdekat di detik-detik terakhir `D`.
10. **Smart Tagging melewatkan restart:** hanya 2 dari 4 `D` punya tag set piece di dekatnya.

### Smart Tagging sebagai sumber possession

11. **Lemah:** tag "menyerang" cocok dengan GT hanya ± 60% frame, tag "bertahan" ± 76%. Tag adalah *fase*, bukan
    possession. → dipakai paling banyak sebagai fitur lemah; **tidak** sebagai proxy kebenaran utama. Proxy metric
    Fase 2 bertumpu pada pseudo-label yang disisihkan dan kewajaran statistik.

**Berikutnya:** `03_bola_mati.ipynb` — detektor `D` (fitur kolektif + Viterbi 2 keadaan) dengan fitur celah bola panjang.